In [14]:
import pandas as pd
import os
from datetime import datetime

# Carpeta donde están los archivos ZQDIS
carpeta = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta Abastos\ZQDIS"

# Carpeta de salida (la misma de siempre)
output_folder = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta Abastos\Outputs"

# Fecha y hora actual para el nombre del archivo
fecha_hora = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join(output_folder, f"ZQDIS_limpio_{fecha_hora}.csv")

# Lista para guardar los DataFrames
dfs = []

# Nombres de columnas válidas para ZQDIS
columnas_validas = ['TpMt','Familia','Denominación','Material','Texto breve de material','UML','Contador','Código EAN','Peso bruto','Un','Volumen','UV']

# Mapeo de UML a Presentación
uml_a_presentacion = {
    'ZPI': 'Inner',
    'ZPM': 'Master',
    'ZPP': 'Pallet',
    'BOL': 'Pieza',
    'CAJ': 'Pieza',
    'CAR': 'Pieza',
    'CB': 'Pieza',
    'JGO': 'Pieza',
    'KG': 'Pieza',
    'M': 'Pieza',
    'PAA': 'Pieza',
    'PZA': 'Pieza',
    'ROL': 'Pieza',
    'SET': 'Pieza'
}

# Itera sobre todos los archivos en la carpeta
for archivo in os.listdir(carpeta):
    ruta = os.path.join(carpeta, archivo)
    if os.path.isfile(ruta) and archivo.lower().endswith('.xls'):
        # Leer solo la primera fila después del salto para saber cuántas columnas hay
        encabezados = pd.read_csv(ruta, sep='\t', skiprows=9, nrows=1, header=None, encoding='latin1')
        n_cols = encabezados.shape[1]
        # Leer el archivo: saltar 3 filas, usar la primera fila después del salto como encabezados, y leer todas las columnas válidas
        df = pd.read_csv(ruta, sep='\t', skiprows=3, header=0, encoding='latin1', usecols=range(n_cols))
        # Quitar espacios al inicio y final de todos los nombres de columnas
        df.columns = [col.strip() for col in df.columns]
        # Filtrar solo las columnas válidas
        df = df[[col for col in columnas_validas if col in df.columns]]
        # Eliminar columnas completamente vacías
        df = df.dropna(axis=1, how='all')
        # Eliminar filas donde la primera columna esté vacía
        if not df.empty:
            df = df.dropna(subset=[df.columns[0]])

        # Procesar columna 'Peso bruto' y convertir a kg según la unidad en 'Un'
        if 'Peso bruto' in df.columns:
            peso_numerico = pd.to_numeric(
                df['Peso bruto'].astype(str).str.replace(',', '', regex=False),
                errors='coerce'
            )
            if 'Un' in df.columns:
                unidad_peso = df['Un'].astype(str).str.strip().str.upper()
                mascara_libras = unidad_peso.isin(['LB', 'LBS', 'LIBRA', 'LIBRAS'])
                peso_numerico.loc[mascara_libras] = peso_numerico.loc[mascara_libras] * 0.45359237
            df['Peso neto (kg)'] = peso_numerico
            df = df.drop(columns=['Peso bruto'])

        # Procesar columna 'Volumen' y convertir a cm3 según la unidad en 'UV'
        if 'Volumen' in df.columns:
            volumen_numerico = pd.to_numeric(
                df['Volumen'].astype(str).str.replace(',', '', regex=False),
                errors='coerce'
            )
            if 'UV' in df.columns:
                unidad_volumen = df['UV'].astype(str).str.strip().str.upper()
                mascara_m3 = unidad_volumen == 'M3'
                mascara_gal = unidad_volumen.isin(['GAL', 'GALON', 'GALONES'])
                volumen_numerico.loc[mascara_m3] = volumen_numerico.loc[mascara_m3] * 1000000
                volumen_numerico.loc[mascara_gal] = volumen_numerico.loc[mascara_gal] * 3785.411784
            df['Volumen(cm3)'] = volumen_numerico
            df = df.drop(columns=['Volumen'])

        # Eliminar columnas de unidad después de las conversiones
        df = df.drop(columns=[col for col in ['Un', 'UV'] if col in df.columns], errors='ignore')

        # Convertir 'Material' y 'Código EAN' a texto y quitar .0 si existe
        for col in ['Material', 'Código EAN']:
            if col in df.columns:
                df[col] = df[col].astype(str).str.replace(r'\.0$', '', regex=True)
        # Reemplazar 'nan' por cadena vacía en 'Código EAN'
        if 'Código EAN' in df.columns:
            df['Código EAN'] = df['Código EAN'].replace('nan', '')
        # Crear columna 'Presentación' a partir de UML
        if 'UML' in df.columns:
            df['Presentación'] = (
                df['UML']
                .astype(str)
                .str.strip()
                .str.upper()
                .map(uml_a_presentacion)
                .fillna('')
            )
        else:
            df['Presentación'] = ''

        # Eliminar filas sin Presentación
        df = df[df['Presentación'].astype(str).str.strip() != '']

        # Eliminar filas con Código EAN vacío cuando la Presentación sea Pieza
        if 'Código EAN' in df.columns:
            mascara_ean_vacio_pieza = (
                (df['Presentación'].astype(str).str.strip().str.upper() == 'PIEZA')
                & (df['Código EAN'].astype(str).str.strip() == '')
            )
            df = df[~mascara_ean_vacio_pieza]

        df['Archivo'] = archivo  # Agrega columna con el nombre del archivo
        dfs.append(df)

# Une todos los DataFrames
if dfs:
    ZQDIS = pd.concat(dfs, ignore_index=True)
    # Eliminar filas duplicadas solo si son exactamente iguales en todas las columnas
    ZQDIS = ZQDIS.drop_duplicates()

    # Renombrar columnas para el archivo final
    ZQDIS = ZQDIS.rename(columns={
        'Material': 'SKU',
        'Contador': 'Piezas por presentación',
        'Volumen(cm3)': 'Volumen (cm3)'
    })

    # Ordenar por Presentación de Z a A
    if 'Presentación' in ZQDIS.columns:
        ZQDIS = ZQDIS.sort_values(by='Presentación', ascending=False)

    print('Columnas detectadas:', list(ZQDIS.columns))
    display(ZQDIS.head())
    # Guarda el resultado en CSV con fecha y hora
    ZQDIS.to_csv(output_path, index=False, encoding='latin1')
    print(f"✅ Archivo ZQDIS generado y guardado en: {output_path}")
else:
    print("No se encontraron archivos válidos en la carpeta ZQDIS.")

C:\Users\igcamposg\AppData\Local\Temp\ipykernel_32040\946918559.py:47: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta, sep='\t', skiprows=3, header=0, encoding='latin1', usecols=range(n_cols))


Columnas detectadas: ['TpMt', 'Familia', 'Denominación', 'SKU', 'Texto breve de material', 'UML', 'Piezas por presentación', 'Código EAN', 'Peso neto (kg)', 'Volumen (cm3)', 'Presentación', 'Archivo']


,TpMt,Familia,Denominación,SKU,Texto breve de material,UML,Piezas por presentación,Código EAN,Peso neto (kg),Volumen (cm3),Presentación,Archivo
91469,ZRFC,P651,Calentadores de agua eléctricos,931136,"R8-CALE-20S Soporte para tubo de 58 mm,",PZA,1,7506240665892,0.100,454.410,Pieza,ZQDIS 09mar.XLS
57326,ZCOM,P521,Accesorios para baño,45222,"RIA-77N Toallero argolla latón satinado,",PZA,1,7506240631316,0.425,4495.248,Pieza,ZQDIS 09mar.XLS
57295,ZCOM,P521,Accesorios para baño,45214,RIA-79 Juego de anclaje para repisa toal,SET,1,7506240631279,0.079,146.160,Pieza,ZQDIS 09mar.XLS
57298,ZCOM,P521,Accesorios para baño,45215,"RIA-75DI Dispensador de jabón de vidrio,",PZA,1,7506240631378,0.300,1419.264,Pieza,ZQDIS 09mar.XLS
24125,ZCOM,P769,Prensas y sargentos,17701,"PRE-6 Prensa de plastico 6""",PZA,1,7501206625330,0.126,871.884,Pieza,ZQDIS 09mar.XLS


✅ Archivo ZQDIS generado y guardado en: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta Abastos\Outputs\ZQDIS_limpio_20260311_093706.csv
